# 🔬 OSFT Fine-Tuning

## What is OSFT (Orthogonal Subspace Fine-Tuning)?

OSFT is a fine-tuning technique that orthogonally decomposes the weight matrix and selectively trains specific components.

### Differences from LoRA

| Category | LoRA | OSFT |
|------|------|------|
| **Approach** | Add low-rank adapter matrices | Orthogonal decomposition of weights with selective unfreezing |
| **Memory** | Additional memory proportional to adapter size | Decomposition + activation + optimizer memory |
| **Parameters** | rank, alpha | unfreeze_rank_ratio |
| **Modularity** | Adapter can be detached | Applied to the full model |
| **Forgetting resistance** | Good due to preserving original weights | Minimizes interference via orthogonal subspaces |

### OSFT Characteristics
- Orthogonal decomposition of weight matrix W: W = U·Σ·V^T
- Only singular values/vectors up to `unfreeze_rank_ratio` are selected as trainable
- The remaining orthogonal subspaces are frozen → advantageous for preserving existing knowledge
- Uses more memory than LoRA, but may offer better forgetting resistance

### Configuration for This Notebook
- **Base model**: `Qwen/Qwen3-4B-Instruct-2507` (same as LoRA)
- **unfreeze_rank_ratio**: 0.25 (trains 25% of total rank)
- **Data**: Same canonical data as LoRA (independent training)

> ⚠️ OSFT runs **independently** from LoRA.  
> It starts from the same base model and the same training data.  
> OSFT is **not** applied on top of a LoRA checkpoint.

In [ ]:
"""Load OSFT config and validate bundle compatibility."""

import os
from pathlib import Path

from rhoai_model_training_lab.config import (
    load_env, load_training_config, load_bundle_config, PROJECT_ROOT,
)
from rhoai_model_training_lab.data import BundleManager

load_env()

# Load OSFT config
osft_config = load_training_config("osft")
model_id = osft_config["model"]["model_id"]
model_revision = osft_config["model"]["model_revision"]

print(f"Model: {model_id} (rev: {model_revision})")
print(f"OSFT unfreeze_rank_ratio: {osft_config['osft']['unfreeze_rank_ratio']}")
print(f"Seed: {osft_config['training']['seed']}")

# Load and validate bundle (same bundle as LoRA)
release_config = load_bundle_config()
bundle_base = release_config.get("bundle", {}).get("base_path", "data/prepared/tau-knowledge-v1")
bundle_path = PROJECT_ROOT / bundle_base

print(f"\nBundle path: {bundle_path}")
mgr = BundleManager.load_bundle(bundle_path)
manifest = mgr.manifest

# Compatibility check (same base model as LoRA)
compat = mgr.validate_compatibility(model_id)
if compat.errors:
    for err in compat.errors:
        print(f"  ❌ {err}")
    raise RuntimeError(
        "Bundle compatibility check failed — cannot proceed with training.\n"
        "Please generate the correct bundle using the data_preparation/ notebooks."
    )

print(f"\n✅ Bundle compatibility check passed")
print(f"   Training samples: {manifest.canonical_train_count} (same as LoRA)")
print(f"   Validation samples: {manifest.canonical_validation_count}")
print(f"   Bundle version: {manifest.bundle_version}")

In [ ]:
"""Preview training data with OSFT-specific formatting."""

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Load OSFT-specific training data
train_data = mgr.get_training_samples("osft", "train")
val_data = mgr.get_training_samples("osft", "validation")

print(f"OSFT training data: {len(train_data)} samples")
print(f"OSFT validation data: {len(val_data)} samples")

# Verify sample parity with LoRA
lora_train = mgr.get_training_samples("lora", "train")
print(f"\n📊 LoRA training samples: {len(lora_train)}")
print(f"   OSFT training samples: {len(train_data)}")
if len(train_data) == len(lora_train):
    print("   ✅ Sample counts match — converted from the same canonical data")
else:
    print("   ⚠️  Sample count mismatch — check backend-specific conversion differences")

# Preview with OSFT formatting focus
print("\n" + "=" * 70)
print("📝 OSFT Training Data Preview")
print("=" * 70)

for i, sample in enumerate(train_data[:2]):
    messages = sample.get("messages", [])
    print(f"\n--- Sample {i+1} ---")
    print(f"Number of messages: {len(messages)}")

    for msg in messages:
        role = msg["role"]
        content = msg.get("content", "") or ""
        is_target = role == "assistant"
        mask_icon = "🎯 [Training target]" if is_target else "🚫 [Masked]"

        display = content[:200] + "..." if len(content) > 200 else content
        print(f"\n  {mask_icon} [{role}]: {display}")

    # Token analysis
    try:
        tokens = tokenizer.apply_chat_template(messages, tokenize=True)
        print(f"\n  Token count: {len(tokens)}")
    except Exception:
        pass

# Memory estimation note
print(f"\n{'=' * 70}")
print("💾 OSFT Memory Notes:")
print(f"  - unfreeze_rank_ratio: {osft_config['osft']['unfreeze_rank_ratio']}")
print("  - OSFT uses more GPU memory than LoRA")
print("  - Includes decomposition + activation + optimizer state")
print(f"  - Batch size: {osft_config['training_args']['per_device_train_batch_size']}")
print(f"  - Gradient accumulation: {osft_config['training_args']['gradient_accumulation_steps']}")

In [ ]:
"""Train using training_hub.osft — actual API call."""

import time
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected.\n"
        "OSFT training requires a CUDA-compatible GPU.\n"
        "Please run this notebook in an environment with a GPU."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / (1024**3):.1f} GB")
print()

# Prepare arguments from config
train_file = str(PROJECT_ROOT / osft_config["data"]["train_file"])
val_file = str(PROJECT_ROOT / osft_config["data"]["validation_file"])
output_dir = str(PROJECT_ROOT / osft_config["training_args"]["output_dir"])

print("=" * 70)
print("🚀 Starting OSFT Training")
print("=" * 70)
print(f"  Training data: {train_file}")
print(f"  Validation data: {val_file}")
print(f"  Output directory: {output_dir}")
print(f"  Epochs: {osft_config['training_args']['num_train_epochs']}")
print(f"  Batch size: {osft_config['training_args']['per_device_train_batch_size']}")
print(f"  Gradient accumulation: {osft_config['training_args']['gradient_accumulation_steps']}")
print(f"  Learning rate: {osft_config['training_args']['learning_rate']}")
print(f"  unfreeze_rank_ratio: {osft_config['osft']['unfreeze_rank_ratio']}")
print()

start_time = time.time()

# Actual training_hub API call
from training_hub import osft as osft_train

training_result = osft_train(
    model_id=model_id,
    model_revision=model_revision,
    train_file=train_file,
    validation_file=val_file,
    output_dir=output_dir,
    unfreeze_rank_ratio=osft_config["osft"]["unfreeze_rank_ratio"],
    num_train_epochs=osft_config["training_args"]["num_train_epochs"],
    per_device_train_batch_size=osft_config["training_args"]["per_device_train_batch_size"],
    gradient_accumulation_steps=osft_config["training_args"]["gradient_accumulation_steps"],
    learning_rate=osft_config["training_args"]["learning_rate"],
    weight_decay=osft_config["training_args"]["weight_decay"],
    warmup_ratio=osft_config["training_args"]["warmup_ratio"],
    lr_scheduler_type=osft_config["training_args"]["lr_scheduler_type"],
    max_seq_length=osft_config["data"]["max_seq_length"],
    bf16=osft_config["training_args"]["bf16"],
    gradient_checkpointing=osft_config["training_args"]["gradient_checkpointing"],
    seed=osft_config["training"]["seed"],
    logging_steps=osft_config["training_args"]["logging_steps"],
    eval_steps=osft_config["training_args"]["eval_steps"],
    save_steps=osft_config["training_args"]["save_steps"],
    save_total_limit=osft_config["training_args"]["save_total_limit"],
    resume_from_checkpoint=osft_config["training_args"].get("resume_from_checkpoint", True),
)

wall_time = time.time() - start_time

print(f"\n✅ OSFT training complete!")
print(f"  Elapsed time: {wall_time/60:.1f} min")
print(f"  Final training loss: {getattr(training_result, 'train_loss', 'N/A')}")
print(f"  Final validation loss: {getattr(training_result, 'eval_loss', 'N/A')}")

In [ ]:
"""Inspect results — loss curve and memory usage comparison."""

import json

# Load training logs
log_history = getattr(training_result, "log_history", None)
output_dir_path = Path(output_dir)

if log_history is None:
    state_path = output_dir_path / "trainer_state.json"
    if state_path.exists():
        with open(state_path) as f:
            state = json.load(f)
        log_history = state.get("log_history", [])

if log_history:
    train_steps = [e["step"] for e in log_history if "loss" in e]
    train_losses = [e["loss"] for e in log_history if "loss" in e]
    eval_steps = [e["step"] for e in log_history if "eval_loss" in e]
    eval_losses = [e["eval_loss"] for e in log_history if "eval_loss" in e]

    try:
        import matplotlib.pyplot as plt

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Loss curve
        axes[0].plot(train_steps, train_losses, label="Training loss", alpha=0.7)
        if eval_losses:
            axes[0].plot(eval_steps, eval_losses, label="Validation loss", marker="o", markersize=4)
        axes[0].set_xlabel("Step")
        axes[0].set_ylabel("Loss")
        axes[0].set_title("OSFT Training Loss Curve")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Memory comparison with LoRA (if available)
        osft_peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
        lora_result_path = PROJECT_ROOT / "checkpoints" / "lora" / "training_result.json"
        lora_peak_vram = 0
        if lora_result_path.exists():
            with open(lora_result_path) as f:
                lora_data = json.load(f)
            lora_peak_vram = lora_data.get("peak_vram_gb", 0)

        methods = ["LoRA", "OSFT"]
        vrams = [lora_peak_vram, osft_peak_vram]
        colors = ["#2196F3", "#FF9800"]
        axes[1].bar(methods, vrams, color=colors)
        axes[1].set_ylabel("Peak VRAM (GB)")
        axes[1].set_title("Peak VRAM Usage Comparison")
        for i, v in enumerate(vrams):
            if v > 0:
                axes[1].text(i, v + 0.1, f"{v:.1f} GB", ha="center")

        plt.tight_layout()
        plt.show()
    except ImportError:
        osft_peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
        print("\nOSFT Training Loss Trend:")
        for step, loss in zip(train_steps[-10:], train_losses[-10:]):
            bar = "█" * int(loss * 20)
            print(f"  Step {step:>6}: {loss:.4f} {bar}")

    # Summary
    osft_peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
    print("\n--- OSFT Training Metrics Summary ---")
    if train_losses:
        print(f"  Initial loss: {train_losses[0]:.4f}")
        print(f"  Final loss: {train_losses[-1]:.4f}")
    if eval_losses:
        print(f"  Final validation loss: {eval_losses[-1]:.4f}")
        print(f"  Best validation loss: {min(eval_losses):.4f}")
    print(f"  Peak VRAM: {osft_peak_vram:.1f} GB")
    print(f"  Training time: {wall_time/60:.1f} min")
else:
    osft_peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
    print("Training logs not found.")

In [ ]:
"""Log OSFT results to MLflow."""

from rhoai_model_training_lab.schemas.training import TrainingResult

osft_result = TrainingResult(
    method="osft",
    model_id=model_id,
    model_revision=model_revision,
    bundle_id=manifest.bundle_name,
    bundle_hash=manifest.bundle_hash,
    seed=osft_config["training"]["seed"],
    train_samples=manifest.canonical_train_count,
    validation_samples=manifest.canonical_validation_count,
    total_steps=train_steps[-1] if train_steps else 0,
    final_train_loss=train_losses[-1] if train_losses else 0.0,
    final_eval_loss=eval_losses[-1] if eval_losses else None,
    best_eval_loss=min(eval_losses) if eval_losses else None,
    wall_time_seconds=wall_time,
    peak_vram_gb=osft_peak_vram,
    gpu_name=torch.cuda.get_device_name(0),
    checkpoint_path=output_dir,
)

mlflow_uri = os.environ.get("MLFLOW_TRACKING_URI", "")
experiment_name = os.environ.get("MLFLOW_EXPERIMENT_TRAINING", "rhoai-model-training-lab-training")

if mlflow_uri:
    try:
        import mlflow

        mlflow.set_tracking_uri(mlflow_uri)
        mlflow.set_experiment(experiment_name)

        with mlflow.start_run(run_name=f"osft-{manifest.bundle_version}") as run:
            mlflow.log_param("method", "osft")
            mlflow.log_param("model_id", model_id)
            mlflow.log_param("bundle_id", manifest.bundle_name)
            mlflow.log_param("bundle_version", manifest.bundle_version)
            mlflow.log_param("unfreeze_rank_ratio", osft_config["osft"]["unfreeze_rank_ratio"])
            mlflow.log_param("learning_rate", osft_config["training_args"]["learning_rate"])
            mlflow.log_param("seed", osft_config["training"]["seed"])

            mlflow.log_metric("final_train_loss", osft_result.final_train_loss)
            if osft_result.final_eval_loss is not None:
                mlflow.log_metric("final_eval_loss", osft_result.final_eval_loss)
            if osft_result.best_eval_loss is not None:
                mlflow.log_metric("best_eval_loss", osft_result.best_eval_loss)
            mlflow.log_metric("wall_time_seconds", osft_result.wall_time_seconds)
            mlflow.log_metric("peak_vram_gb", osft_result.peak_vram_gb)
            mlflow.log_metric("total_steps", osft_result.total_steps)
            mlflow.log_metric("train_samples", osft_result.train_samples)

            osft_result.mlflow_run_id = run.info.run_id
            osft_result.mlflow_experiment = experiment_name
            print(f"✅ MLflow logging complete (run_id: {run.info.run_id})")

    except Exception as exc:
        print(f"⚠️  MLflow logging failed: {exc}")
        print("  Training results have been saved locally.")
else:
    print("⚠️  MLFLOW_TRACKING_URI not set — saving results locally only.")

# Save result locally
result_path = Path(output_dir) / "training_result.json"
result_path.parent.mkdir(parents=True, exist_ok=True)
with open(result_path, "w") as f:
    f.write(osft_result.model_dump_json(indent=2))
print(f"📄 Training result saved: {result_path}")

In [ ]:
"""Verify OSFT checkpoint."""

from rich.table import Table
from rich.console import Console

console = Console()

checkpoint_dir = Path(output_dir)
checks = []

# Check for model files
has_model_files = (
    list(checkpoint_dir.glob("*.safetensors"))
    or list(checkpoint_dir.glob("*.bin"))
    or list(checkpoint_dir.glob("model*.safetensors"))
)
checks.append(("Model weight files", bool(has_model_files)))

# Check config
has_config = (checkpoint_dir / "config.json").exists()
checks.append(("Model config file", has_config))

# Check tokenizer
has_tokenizer = (checkpoint_dir / "tokenizer_config.json").exists()
checks.append(("Tokenizer files", has_tokenizer))

# Try reloading
reload_ok = False
try:
    from transformers import AutoModelForCausalLM

    # Just verify the config can be loaded (don't load full model)
    from transformers import AutoConfig
    cfg = AutoConfig.from_pretrained(str(checkpoint_dir), trust_remote_code=True)
    reload_ok = True
    print(f"✅ Model config reload succeeded: {cfg.model_type}")
except Exception as exc:
    print(f"⚠️  Model config reload failed: {exc}")

checks.append(("Model reload verification", reload_ok))

# Checkpoint size
total_size = sum(f.stat().st_size for f in checkpoint_dir.rglob("*") if f.is_file())
size_gb = total_size / (1024**3)
checks.append((f"Checkpoint size ({size_gb:.2f} GB)", size_gb > 0))

# Summary table
table = Table(title="🔍 OSFT Checkpoint Verification", show_header=True)
table.add_column("Item", style="bold")
table.add_column("Status")

for name, ok in checks:
    status = "✅" if ok else "❌"
    table.add_row(name, status)

console.print(table)

all_ok = all(ok for _, ok in checks)
if all_ok:
    print("\n🎉 OSFT training completed successfully!")
    print("\nNext steps:")
    print("  📓 05_export_and_deploy.ipynb — Model export and deployment")
    print("  Both LoRA and OSFT checkpoints are ready, so comparative evaluation is possible.")
else:
    print("\n⚠️  Some verification items failed. Please review the results above.")